# Week 5 - Joint Gaussians

**Artifact:** `notebooks/w05_joint_gaussians.ipynb`  
**Audience:** after Week 5 S1/S2 on joint, marginal, conditional distributions, covariance, and correlation.  
**Goal:** connect those definitions to the special case that diffusion models use constantly: a joint Gaussian whose conditional distribution is still Gaussian.

This notebook is intentionally a TODO baseline. The `_solved` copy starts identical; do your work there and keep this one as the clean scaffold.

## Outline

1. Build and sample a correlated 2D Gaussian from a covariance matrix.
2. Visualize covariance through scatter plots and density contours.
3. Inspect empirical conditional slices $Y \mid X \approx x_0$.
4. Implement the analytic conditional Gaussian formula.
5. Compare analytic conditional means/variances against empirical slice estimates.
6. Check that each conditional slice is well described by a 1D Gaussian.

---
## Setup

We only need NumPy and Matplotlib. The RNG is seeded so your empirical checks should be stable up to small Monte Carlo noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
plt.style.use("seaborn-v0_8-whitegrid")

np.set_printoptions(precision=3, suppress=True)

---
## 1. Correlated 2D Gaussian From a Covariance Matrix

For a 2D Gaussian vector

$$
\begin{bmatrix}X \\ Y\end{bmatrix}
\sim \mathcal{N}\!\left(
\begin{bmatrix}\mu_X \\ \mu_Y\end{bmatrix},
\begin{bmatrix}
\sigma_X^2 & \rho\sigma_X\sigma_Y \\
\rho\sigma_X\sigma_Y & \sigma_Y^2
\end{bmatrix}
\right),
$$

the off-diagonal entries encode covariance. The normalized version is the correlation $\rho$.

In [ ]:
mu = np.array([0.5, -0.25])
sigma_x = 1.2
sigma_y = 0.75
rho = 0.72

Sigma = np.array(
    [
        [sigma_x**2, rho * sigma_x * sigma_y],
        [rho * sigma_x * sigma_y, sigma_y**2],
    ]
)

# A covariance matrix must be symmetric positive definite.
eigenvalues = np.linalg.eigvalsh(Sigma)
assert np.all(eigenvalues > 0), eigenvalues

n = 80_000
samples = rng.multivariate_normal(mean=mu, cov=Sigma, size=n)
x = samples[:, 0]
y = samples[:, 1]

sample_mean = samples.mean(axis=0)
sample_cov = np.cov(samples, rowvar=False)
sample_corr = np.corrcoef(samples, rowvar=False)[0, 1]

print("Target mean:", mu)
print("Sample mean:", sample_mean)
print("Target covariance:\n", Sigma)
print("Sample covariance:\n", sample_cov)
print("Target correlation:", rho)
print("Sample correlation:", round(sample_corr, 3))

assert np.allclose(sample_mean, mu, atol=0.03)
assert np.allclose(sample_cov, Sigma, atol=0.04)
assert abs(sample_corr - rho) < 0.03

The scatter should form a tilted ellipse: positive correlation means larger $X$ values tend to come with larger $Y$ values.

In [ ]:
def bivariate_normal_pdf(grid_points, mean, cov):
    """Evaluate a 2D Gaussian density on points with final dimension 2."""
    diff = grid_points - mean
    inv_cov = np.linalg.inv(cov)
    exponent = np.einsum("...i,ij,...j->...", diff, inv_cov, diff)
    normalizer = 2 * np.pi * np.sqrt(np.linalg.det(cov))
    return np.exp(-0.5 * exponent) / normalizer

x_grid = np.linspace(mu[0] - 4 * sigma_x, mu[0] + 4 * sigma_x, 180)
y_grid = np.linspace(mu[1] - 4 * sigma_y, mu[1] + 4 * sigma_y, 180)
xx, yy = np.meshgrid(x_grid, y_grid)
grid = np.stack([xx, yy], axis=-1)
zz = bivariate_normal_pdf(grid, mu, Sigma)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(x[:3500], y[:3500], s=6, alpha=0.18, color="tab:blue", edgecolor="none")
ax.contour(xx, yy, zz, levels=8, colors="black", linewidths=1.0, alpha=0.75)
ax.scatter(mu[0], mu[1], marker="x", s=90, color="tab:red", label="mean")
ax.set(title="Correlated 2D Gaussian", xlabel="X", ylabel="Y")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()

---
## 2. Conditional Slices

For continuous variables, conditioning on exactly $X=x_0$ has probability zero. In samples, we approximate a conditional slice with a thin band:

$$
Y \mid X \approx x_0 \quad \leadsto \quad Y \text{ values where } |X - x_0| \le \text{bandwidth}.
$$

As the bandwidth shrinks and the sample size grows, the slice estimate gets closer to the true conditional distribution.

In [ ]:
def empirical_slice_stats(x_values, y_values, x0, bandwidth):
    """Estimate conditional moments from a thin vertical slice around x0."""
    mask = np.abs(x_values - x0) <= bandwidth
    y_slice = y_values[mask]
    return {
        "x0": x0,
        "count": int(mask.sum()),
        "mean": float(y_slice.mean()),
        "var": float(y_slice.var(ddof=1)),
        "values": y_slice,
    }

x_slices = np.array([mu[0] - 1.4, mu[0], mu[0] + 1.4])
bandwidth = 0.06
slice_stats = [empirical_slice_stats(x, y, x0, bandwidth) for x0 in x_slices]

assert all(stat["count"] > 500 for stat in slice_stats)
[(round(stat["x0"], 2), stat["count"], round(stat["mean"], 3), round(stat["var"], 3)) for stat in slice_stats]

In [ ]:
fig, axes = plt.subplots(1, len(slice_stats), figsize=(13, 3.5), sharey=True)

for ax, stat in zip(axes, slice_stats):
    ax.hist(stat["values"], bins=35, density=True, alpha=0.75, color="tab:green")
    ax.axvline(stat["mean"], color="black", linewidth=2, label="empirical mean")
    ax.set_title(f"Y | X approx {stat['x0']:.2f}\nn={stat['count']}")
    ax.set_xlabel("Y")
    ax.legend(fontsize=8)

axes[0].set_ylabel("density")
fig.suptitle("Empirical conditional slices")
fig.tight_layout()
plt.show()

**Observation checkpoint.** As $x_0$ moves left to right, the center of the $Y$ slice should move in the same direction because $\rho > 0$. The spread of the slices should stay roughly constant.

---
## 3. Analytic Conditional Gaussian Formula

For a bivariate Gaussian, the conditional distribution is Gaussian:

$$
Y \mid X=x \sim \mathcal{N}\!\left(
\mu_Y + \frac{\Sigma_{YX}}{\Sigma_{XX}}(x-\mu_X),
\Sigma_{YY} - \frac{\Sigma_{YX}\Sigma_{XY}}{\Sigma_{XX}}
\right).
$$

In correlation form, the conditional mean is a line and the conditional variance does not depend on $x$:

$$
\mathbb{E}[Y\mid X=x]
= \mu_Y + \rho\frac{\sigma_Y}{\sigma_X}(x-\mu_X),
\qquad
\mathrm{Var}(Y\mid X=x)=\sigma_Y^2(1-\rho^2).
$$

### TODO 1: Implement Conditional Parameters

Fill in the function below in your solved notebook. Do not estimate from samples here; this should use only `mu`, `Sigma`, and `x_value`.

In [ ]:
def conditional_y_given_x(x_value, mean, cov):
    """Return analytic mean and variance of Y | X = x_value for a 2D Gaussian."""
    mu_x, mu_y = mean
    sigma_xx = cov[0, 0]
    sigma_xy = cov[0, 1]
    sigma_yx = cov[1, 0]
    sigma_yy = cov[1, 1]

    # TODO: compute the conditional mean using the formula above.
    cond_mean = np.nan

    # TODO: compute the conditional variance using the formula above.
    cond_var = np.nan

    return float(cond_mean), float(cond_var)

# Optional self-check after completing the TODO:
# m0, v0 = conditional_y_given_x(mu[0], mu, Sigma)
# assert np.isclose(m0, mu[1])
# assert 0 < v0 < Sigma[1, 1]

### TODO 2: Verify Analytic Mean Empirically

Use the empirical slice estimates from Section 2 and compare them to your analytic conditional mean. Expect small mismatch because a slice uses $X \approx x_0$, not exactly $X=x_0$.

In [ ]:
comparison_rows = []

for stat in slice_stats:
    x0 = stat["x0"]
    empirical_mean = stat["mean"]
    empirical_var = stat["var"]

    # TODO: call your analytic function once TODO 1 is complete.
    analytic_mean = np.nan
    analytic_var = np.nan
    # analytic_mean, analytic_var = conditional_y_given_x(x0, mu, Sigma)

    comparison_rows.append(
        {
            "x0": round(x0, 3),
            "empirical_mean": round(empirical_mean, 3),
            "analytic_mean": analytic_mean,
            "empirical_var": round(empirical_var, 3),
            "analytic_var": analytic_var,
            "slice_count": stat["count"],
        }
    )

comparison_rows

# Optional self-check after completing the TODO:
# for row in comparison_rows:
#     assert abs(row["empirical_mean"] - row["analytic_mean"]) < 0.08
#     assert abs(row["empirical_var"] - row["analytic_var"]) < 0.08

---
## 4. Conditional Slices Look Gaussian

Even before using the analytic formula, each empirical slice can be compared to a Gaussian fitted with that slice's own sample mean and variance. This is an empirical check of the statement:

$$
\text{conditional of a joint Gaussian is Gaussian.}
$$

After completing TODO 1, repeat the overlay with analytic parameters instead of fitted slice parameters.

In [ ]:
def normal_pdf(values, mean, var):
    std = np.sqrt(var)
    return np.exp(-0.5 * ((values - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))

fig, axes = plt.subplots(1, len(slice_stats), figsize=(13, 3.5), sharey=True)

for ax, stat in zip(axes, slice_stats):
    y_slice = stat["values"]
    y_line = np.linspace(np.percentile(y_slice, 0.5), np.percentile(y_slice, 99.5), 240)
    fitted_pdf = normal_pdf(y_line, stat["mean"], stat["var"])

    ax.hist(y_slice, bins=35, density=True, alpha=0.65, color="tab:purple", label="slice histogram")
    ax.plot(y_line, fitted_pdf, color="black", linewidth=2, label="Gaussian fit")
    ax.set_title(f"X approx {stat['x0']:.2f}")
    ax.set_xlabel("Y")
    ax.legend(fontsize=8)

axes[0].set_ylabel("density")
fig.suptitle("Empirical conditional slices are close to Gaussian")
fig.tight_layout()
plt.show()

### TODO 3: Replace the Fitted Overlay With the Analytic Conditional Gaussian

Create the same plot again, but use `conditional_y_given_x(x0, mu, Sigma)` for the overlay parameters.

Checklist:

- The analytic mean should shift linearly with $x_0$.
- The analytic variance should be the same for every $x_0$.
- The analytic Gaussian should track the histogram better as sample size increases and bandwidth decreases.

In [ ]:
# TODO: build the analytic overlay plot here after TODO 1 is complete.
# Hint: reuse normal_pdf, slice_stats, and conditional_y_given_x.

fig, ax = plt.subplots(figsize=(6, 3))
ax.axis("off")
ax.text(
    0.02,
    0.65,
    "TODO: overlay analytic conditional PDFs on the slice histograms",
    fontsize=11,
)
plt.show()

---
## Common Pitfall

Do not confuse these two quantities:

- $\mathbb{E}[Y \mid X=x]$: a deterministic function of the chosen conditioning value $x$.
- A sample mean from a band $|X-x|\le h$: a Monte Carlo estimate that depends on sample size and bandwidth.

The first is an analytic property of the distribution. The second is a numerical approximation.

## Optional Extension

Repeat the experiment with `rho = 0.0`, `rho = -0.7`, and `rho = 0.95`.

Questions to check:

- What happens to the conditional mean slope?
- What happens to the conditional variance as $|\rho|$ approaches 1?
- Why does a very thin ellipse imply less uncertainty in $Y \mid X=x$?

## Takeaways

- A covariance matrix controls the geometry of a 2D Gaussian.
- Correlation tilts the joint density contours.
- Slicing a joint Gaussian at $X=x$ gives a 1D Gaussian in $Y$.
- The conditional mean is linear in $x$ and the conditional variance is constant in $x$.
- This is the algebraic backbone of Gaussian conditioning steps used later in denoising and diffusion derivations.